**Day 1 exploration — retained for context.** These cells and saved results are not the current January 2008 forecasting workflow. Same-season traits and weather are unavailable at that decision; within-population validation does not test new 2008 populations. Average testcross yield is a proxy for breeding merit, not automatically pure GCA.

Use the repository-root README and `python scripts/run_pipeline.py --sample` for the corrected workflow. See [notebook context](README.md) and [verification report](../report/CODE_ALIGNMENT.md).

In [2]:
import pandas as pd

pheno = pd.read_csv("../data/raw/sample_data/sample_C1_phenotype_100rows.csv")
env = pd.read_csv("../data/raw/sample_data/sample_environmental_20rows.csv")
geno = pd.read_csv("../data/raw/sample_data/sample_C1.1_100rows.csv", index_col=0)

In [3]:
print("Phenotype:", pheno.shape)
print("Environment:", env.shape)
print("Genomic:", geno.shape)

Phenotype: (100, 33)
Environment: (20, 86)
Genomic: (100, 2911)


In [4]:
pheno.head()

,Unnamed: 0,projects_x,ProjectID,shorthand_x,YEAR_x,LOC,LONGITUDE,LATITUDE,LINE,ERM,...,shorthand_y,Unnamed: 0_y,MAB_PROJECT_ID,YEAR_y,GERMPLASM_ID,GENERATION_NAME,HG,FILE_LIST,CROSS,GERMPLASM_ID_TESTER
0,0,project209265_year2001.Rdata,209265,C1.1,2001,NEDA,-96.41,42.42,191,NaN,...,C1.1,1311,209265,2001,1589589,F2,Cluster1,project209265_year2001.Rdata,1589589/200761,1376340.0
1,1,project209265_year2001.Rdata,209265,C1.1,2001,NEDA,-96.41,42.42,193,NaN,...,C1.1,1311,209265,2001,1589589,F2,Cluster1,project209265_year2001.Rdata,1589589/200761,1376340.0
2,2,project209265_year2001.Rdata,209265,C1.1,2001,IAPR,-95.62,43.08,62,102.066,...,C1.1,1311,209265,2001,1589589,F2,Cluster1,project209265_year2001.Rdata,1589589/200761,1376340.0
3,3,project209265_year2001.Rdata,209265,C1.1,2001,IADA,-94.06,42.26,160,104.683,...,C1.1,1311,209265,2001,1589589,F2,Cluster1,project209265_year2001.Rdata,1589589/200761,1376340.0
4,4,project209265_year2001.Rdata,209265,C1.1,2001,IAPR,-95.62,43.08,61,102.430,...,C1.1,1311,209265,2001,1589589,F2,Cluster1,project209265_year2001.Rdata,1589589/200761,1376340.0


In [5]:
env.head()

,YEAR,LOC,X04_PRCP,X05_PRCP,X06_PRCP,X07_PRCP,X08_PRCP,X09_PRCP,X10_PRCP,X04_TAVG,...,phh2o_15_30cm,phh2o_30_60cm,phh2o_60_100cm,phh2o_100_200cm,soc_0_5cm,soc_5_15cm,soc_15_30cm,soc_30_60cm,soc_60_100cm,soc_100_200cm
0,2001,SDEP,93.1,80.8,49.2,97.4,55.6,89.4,19.300000,9.098235,...,71.0,74.0,78.0,79.0,292.0,190.0,124.0,74.0,46.0,27.0
1,2002,SDEP,77.5,74.4,54.7,46.3,180.7,23.9,67.900000,9.541765,...,71.0,74.0,78.0,79.0,292.0,190.0,124.0,74.0,46.0,27.0
2,2003,SDEP,80.3,96.6,162.2,30.5,19.1,197.1,17.000000,9.234118,...,71.0,74.0,78.0,79.0,292.0,190.0,124.0,74.0,46.0,27.0
3,2004,SDEP,61.8,157.0,80.8,40.8,43.2,118.9,50.041176,9.372353,...,71.0,74.0,78.0,79.0,292.0,190.0,124.0,74.0,46.0,27.0
4,2005,SDEP,103.9,92.7,165.2,84.9,76.7,136.7,18.500000,10.726471,...,71.0,74.0,78.0,79.0,292.0,190.0,124.0,74.0,46.0,27.0


Let's see the first 5 rows, first 10 SNP columns

In [6]:
geno.iloc[:5, :10]

,M00003409443,M00000005000,M00000226152,M00000038753,M00009474085,M00003426327,M00009437705,M00009450083,M00003679000,M00009437330
PID200761,1,1,1,-1,1.0,-1,1,1,1,1
PID1589589,1,1,1,-1,-1.0,-1,1,1,1,1
00000000001,1,1,1,-1,NaN,-1,1,1,1,1
00000000002,1,1,1,-1,NaN,-1,1,1,1,1
00000000003,1,1,1,-1,NaN,-1,1,1,1,1


Some notes: the phenotype has 100 rows, 33 columns and the col names are messy as it has you can see YEAR_x, YEAR_y, projects_x, projects_y, Unnamed: 0_x, Unnamed: 0_y. These _x and _y suffixes mean this file was already merged from two sources before we got it & dup. columns got renamed automatically. Drop those later before modeling. The ones we actually care about are LOC, LINE, YLD_BE, LINE_UNIQUE_ID, CROSS, GERMPLASM_ID_TESTER, and the trait columns.

Environment has 20 rows, 86 columns and is very wide with monthly precipitation, temperature, and soil measurements at multiple depths. Only 20 rows in the sample so this is obviously just a slice. The soil columns like soc_0_5cm (soil organic carbon) are constant across all 5 visible rows because it's the same location (SDEP) across multiple years which makes sense as soil doesn't change year to year.

Genomic has 100 rows, 2911 columns. In terms of what stands out, first, rows 1 and 2 are our parents PID200761, PID1589589 exactly as expected, followed by progeny 00000000001 onward. Second, for column M00009474085 the parents have values (1.0 and -1.0) but all progeny show NaN. That's a marker where imputation failed or the data is missing for the offspring. We would need to handle missing SNPs!

In [7]:
#how many missing values per column in phenotype?
print(pheno.isnull().sum()[pheno.isnull().sum() > 0])

ERM       46
MST        3
PHT       56
RTLP      44
STLP      54
TWT        2
YLD_BE     3
EHT       57
dtype: int64


In [8]:
#what % of SNPs have any missing values?
missing_snp_pct = geno.isnull().any(axis=0).mean() * 100
print(f"{missing_snp_pct:.1f}% of SNP columns have at least one missing value")

32.3% of SNP columns have at least one missing value


In [9]:
#which phenotype columns are actually useful vs junk?
print(pheno.columns.tolist())

['Unnamed: 0', 'projects_x', 'ProjectID', 'shorthand_x', 'YEAR_x', 'LOC', 'LONGITUDE', 'LATITUDE', 'LINE', 'ERM', 'MST', 'PHT', 'RTLP', 'STLP', 'TWT', 'YLD_BE', 'EHT', 'SET', 'CLUSTER', 'LINE_UNIQUE_ID', 'Unnamed: 0_x', 'projects_y', 'projectID', 'shorthand_y', 'Unnamed: 0_y', 'MAB_PROJECT_ID', 'YEAR_y', 'GERMPLASM_ID', 'GENERATION_NAME', 'HG', 'FILE_LIST', 'CROSS', 'GERMPLASM_ID_TESTER']


the good news is YLD_BE yield only has 3 missing out of 100, so the target variable is mostly intact. The bad news is EHT or ear height, PHT or plant height, ERM, RTLP, STLP all have 40-57% missing so over half the rows. For now that's fine since we're just exploring sample data, but it means those traits can't be used as predictors without imputation in the full dataset.

32.3% SNP missingness which is the sample data showing raw non-imputed SNPs. The full genomic files in data/raw/genotypes/ are the _Imputed.csv versions, so this problem mostly goes away when we switch to full data. But good to know the pattern now.

## Join Key Verification
Check that the three tables can actually be linked before building the merge pipeline.

In [14]:
#---- Pheno -> Geno join key ----
#LINE_UNIQUE_ID format: 'C1.1_2001_191' -> last segment = line number
pheno['LINE_NUM'] = pheno['LINE_UNIQUE_ID'].str.extract(r'(\d+)$').astype(int)
#geno row index: '00000000191' -> strip leading zeros -> int
#parents (PID...) are non-numeric; drop them for the overlap check
geno_idx_int = pd.to_numeric(geno.index, errors='coerce').dropna().astype(int)

pheno_lines  = set(pheno['LINE_NUM'])
geno_lines   = set(geno_idx_int)

overlap_pg   = pheno_lines & geno_lines
only_pheno   = pheno_lines - geno_lines
only_geno    = geno_lines  - pheno_lines

print('=== Pheno <-> Geno (LINE_NUM) ===')
print(f'  Pheno lines  : {len(pheno_lines)}')
print(f'  Geno lines   : {len(geno_lines)}')
print(f'  Overlap      : {len(overlap_pg)}')
print(f'  Only in pheno: {sorted(only_pheno)}')
print(f'  Only in geno : {sorted(only_geno)[:10]} ...')


=== Pheno <-> Geno (LINE_NUM) ===
  Pheno lines  : 80
  Geno lines   : 98
  Overlap      : 38
  Only in pheno: [106, 107, 108, 109, 110, 111, 112, 114, 115, 117, 118, 128, 132, 134, 138, 141, 142, 143, 145, 146, 147, 148, 152, 158, 159, 160, 161, 164, 165, 179, 181, 182, 183, 185, 186, 187, 189, 191, 192, 193, 197, 198]
  Only in geno : [1, 4, 7, 9, 10, 12, 13, 14, 15, 17] ...


In [15]:
#---- Pheno -> Env join key ----
#Rename YEAR_x -> YEAR first (YEAR_y is the dup from the pre-merged source)
pheno_clean = pheno.rename(columns={'YEAR_x': 'YEAR'})

pheno_loc_year = set(zip(pheno_clean['LOC'], pheno_clean['YEAR']))
env_loc_year   = set(zip(env['LOC'],          env['YEAR']))

overlap_pe   = pheno_loc_year & env_loc_year
only_p       = pheno_loc_year - env_loc_year
only_e       = env_loc_year   - pheno_loc_year

print('=== Pheno <-> Env (LOC + YEAR) ===')
print(f'  Pheno (LOC,YEAR) pairs : {len(pheno_loc_year)}')
print(f'  Env   (LOC,YEAR) pairs : {len(env_loc_year)}')
print(f'  Overlap                : {len(overlap_pe)}')
print(f'  Only in pheno          : {sorted(only_p)}')
print(f'  Only in env            : {sorted(only_e)}')


=== Pheno <-> Env (LOC + YEAR) ===
  Pheno (LOC,YEAR) pairs : 6
  Env   (LOC,YEAR) pairs : 20
  Overlap                : 0
  Only in pheno          : [('IADA', 2001), ('IAFO', 2001), ('IAPR', 2001), ('IASP', 2001), ('MNOW', 2001), ('NEDA', 2001)]
  Only in env            : [('IALI', 2006), ('IALI', 2007), ('IALI', 2008), ('MNHE', 2007), ('MNHE', 2008), ('MNKA', 2008), ('MOHA', 2000), ('MOHA', 2001), ('MOHA', 2002), ('MOHA', 2004), ('MOHA', 2005), ('MOHA', 2006), ('MOHA', 2008), ('NCFV', 2007), ('SDEP', 2001), ('SDEP', 2002), ('SDEP', 2003), ('SDEP', 2004), ('SDEP', 2005), ('WIEG', 2006)]


**Expected outcomes (sample data):**
- Pheno<->Geno: most progeny lines should overlap; the two parent rows (PID...) in geno will be non-numeric and drop out cleanly.
- Pheno<->Env: the 100-row pheno slice covers a handful of (LOC, YEAR) combos; the 20-row env slice may not cover all of them — that's expected in sample mode and will resolve on full data.

Pheno <-> Geno (38/80 overlap): The sample geno file is C1.1_100rows.csv, which is genomic file #1 and covers progeny lines roughly in the 1-100 range. The pheno sample happens to contain lines 106-198 so on the high end. So the two slices are mostly from different parts of the population and just barely overlap at the boundary. On full data with all 500 geno files, every line in pheno should have a geno match.

Pheno <-> Env (0 overlap): The pheno sample is 100 rows all from year 2001 at Iowa/Minnesota/Nebraska locations (IADA, IAFO, IAPR, IASP, MNOW, NEDA). The env sample's 20 rows were sliced from entirely different locations (SDEP, MOHA, MNHE, etc.). The slices just don't intersect. No actual join key mismatch, just independent slicing. On full data every pheno (LOC, YEAR) pair will have an env row.

One thing worth confirming before preprocessing: the env join is a left join on pheno, so on full data any pheno (LOC, YEAR) that genuinely has no env row would get NaN env features rather than being dropped. Worth a quick pheno_merge.isnull().sum() check after the merge in 02_preprocessing to catch any real gaps.